# Moebius — Sphere-Congruence Quad Mesh Pipeline

Full pipeline in six stages:
0. **Setup** — imports, env check
1. **Load surface** — B-spline JSON, u/v range, flip options
2. **Initialisation** — sphere radii r(u,v), line congruence l(u,v)
3. **LC optimisation** — minimise l·cu, l·cv; enforce angle with n
4. **Torsal optimisation** — add torsal-plane constraints
5. **Remeshing** — export cross-field, run remesher, load result
6. **Post-optimisation** — sphere fit, support planarity, face regularity
7. **Save results** — OBJ files, sphere panels, state pickle

In [41]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection, compute_init,
    setup_bspline_optimizer, run_bspline_optimizer,
    setup_lc_optimizer, run_lc_optimizer,
    setup_torsal_optimizer, run_torsal_optimizer, recompute_torsal_directions,
    check_remesher, export_frame_field, torsal_field_per_quad,
    run_remesher, load_remeshed_surface, remesh_quality, sweep_gradient,
    setup_postopt_optimizer, run_postopt_optimizer,
    save_results, load_state,
)
from hanan.geometry.io import read_obj, write_obj
from moebius.utils.bsplines import sample_bspline_surface, bspline_curvatures
from moebius.glyphs import spherical_panels_from_mesh, sphere_centers_radii_from_params
from scipy.interpolate import bisplev

kv.init()
kv.clear()

---
## 1 — Load B-spline surface

In [42]:
# ── parameters (edit these) ───────────────────────────────────────────────────
SURFACE_NAME    = "Tunel"          # JSON filename without extension
EXPERIMENT_NAME = "experiment0"

# u/v sampling
U_SAMPLE = 25
V_SAMPLE = 25
INTERVAL_U = [0.1, 0.9]
INTERVAL_V = [0.1, 0.9]

# Orientation options
FLIP_NORMALS = False   # reverse u direction
SWAP_UV      = False   # swap u <-> v
# ─────────────────────────────────────────────────────────────────────────────

state = load_surface(
    SURFACE_NAME, EXPERIMENT_NAME,
    u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE,
    interval_u=INTERVAL_U, interval_v=INTERVAL_V,
)

# Apply flip/swap if requested
if FLIP_NORMALS:
    cp = state.bspline.controlpoints
    state.bspline.controlpoints = cp[::-1, :, :]
    state.normalsFlipped = True
if SWAP_UV:
    state.bspline = state.bspline.clone()
    state.bspline.controlpoints = state.bspline.controlpoints.swapaxes(0, 1)
    state.uvSwapped = True

print(f"Surface: {SURFACE_NAME}")
print(f"Sampling: {U_SAMPLE}×{V_SAMPLE} on u∈{INTERVAL_U}, v∈{INTERVAL_V}")

Surface: Tunel
Sampling: 25×25 on u∈[0.1, 0.9], v∈[0.1, 0.9]


In [31]:
import ipywidgets as widgets
from IPython.display import display

def _update_surface(u_min, u_max, v_min, v_max, u_n, v_n):
    if u_min >= u_max or v_min >= v_max:
        return
    sample_selection(state,
                     interval_u=[u_min, u_max],
                     interval_v=[v_min, v_max],
                     u_sample_num=u_n,
                     v_sample_num=v_n)
    V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
    K, H, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
    kv.register_surface_mesh("surface", V, F)
    kv.add_scalar_quantity("surface", "MeanCurvature",     H.ravel(), defined_on="vertices")
    kv.add_scalar_quantity("surface", "GaussianCurvature", K.ravel(), defined_on="vertices")

_sl = dict(continuous_update=False, style={"description_width": "80px"}, layout=widgets.Layout(width="340px"))
u_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="u min", **_sl)
u_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="u max", **_sl)
v_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="v min", **_sl)
v_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="v max", **_sl)
u_n_w   = widgets.IntSlider(value=U_SAMPLE, min=5, max=60, step=5, description="u samples", **_sl)
v_n_w   = widgets.IntSlider(value=V_SAMPLE, min=5, max=60, step=5, description="v samples", **_sl)

out = widgets.interactive_output(_update_surface, {
    "u_min": u_min_w, "u_max": u_max_w,
    "v_min": v_min_w, "v_max": v_max_w,
    "u_n":   u_n_w,   "v_n":   v_n_w,
})
ui = widgets.VBox([
    widgets.HBox([u_min_w, u_max_w]),
    widgets.HBox([v_min_w, v_max_w]),
    widgets.HBox([u_n_w,   v_n_w]),
])
display(ui, out)
kv.show()

Output()

[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 1b — B-spline optimisation (enforce H ≠ 0)

Slightly deforms the surface control points to ensure the mean curvature H has
a consistent nonzero sign throughout the domain. This makes the sphere-congruence
initialisation r = 1/H well-conditioned.

Skip this cell if the surface already has |H| > 0 everywhere.

In [28]:
from moebius.pipeline import setup_bspline_optimizer, run_bspline_optimizer

# ── parameters ────────────────────────────────────────────────────────────────
H_THRESH     = 0.05
W_H          = 1.0
W_STEP       = 0.1
BSPLINE_ITER = 30
# ─────────────────────────────────────────────────────────────────────────────

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_before, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H before  min={H_before.min():.4f}  max={H_before.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_before.ravel())) != 0).sum())}")

setup_bspline_optimizer(state, H_thresh=H_THRESH, w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=BSPLINE_ITER)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_after, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H after   min={H_after.min():.4f}  max={H_after.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_after.ravel())) != 0).sum())}")

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "H_before", H_before.ravel(), defined_on="vertices")
kv.add_scalar_quantity("surface", "H_after",  H_after.ravel(),  defined_on="vertices")
kv.show()

H before  min=1.3549  max=3.9857  zero-crossings≈0
Thread pool: 3 workers (terms=3, cpus=32)
Starting fixed-damping LM optimization (max_iter=30, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 4.487863e-04  | 0.000000e+00  | 1.085904e-02  | 1.000000e-06  | N/A      | 0.0092  


KeyboardInterrupt: 

---
## 2 — Sphere-congruence initialisation

In [44]:
state.init_mode = 0

compute_init(state)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
n_flat = state.n_surf.reshape(-1, 3)
Vc = V + r_vals.ravel()[:, None] * n_flat

kv.register_surface_mesh("sphere_centers_init", Vc, F)
kv.register_surface_mesh("surface", V, F)
kv.add_vector_quantity("surface", "l_init", state.l_init.reshape(-1, 3), defined_on="vertices")
kv.show()

Init  r range  [0.2456, 0.3778]
Init  l·n mean  0.9590
[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 3a — Line-congruence optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
LC_MAX_ITER  = 100
ANGLE_LC_DEG = 15.0   # theta: MAX angle l <-> n. Smaller = stronger.
                      # Paper Tables 1-2 use 15 deg (25 deg for the denser grid).
W_LC         = 1.0
W_ORTH       = 2.0    # paper Table 1: w_LCOrth = 2 in this stage
HINGE        = True   # one-sided residuals, no mu/theta slack variables
# ─────────────────────────────────────────────────────────────────────────────

state.angle_lc = ANGLE_LC_DEG

setup_lc_optimizer(state, w_lc=W_LC, w_orth=W_ORTH, hinge=HINGE)
run_lc_optimizer(state, max_iter=LC_MAX_ITER)

In [47]:
l_opt, cp_opt = state.lc_opt.unpack("l", "rij")
state.r_uv[2] = cp_opt
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
Vc = V + r_vals.ravel()[:, None] * n_flat

l_opt   = l_opt.reshape(-1, 3)
angle_n = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", l_opt, n_flat), -1, 1)))

kv.register_surface_mesh("surface",        V, F)
kv.register_surface_mesh("sphere_centers", Vc, F)
kv.add_vector_quantity("surface", "l_opt",    l_opt,   defined_on="vertices")
kv.add_scalar_quantity("surface", "angle_l_n", angle_n, defined_on="vertices")
kv.show()

[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 3b — Torsal optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
TORSAL_MAX_ITER  = 100
ANGLE_TORSAL_DEG = 75.0   # alpha: MINIMUM angle between the torsal PLANES.
                          # Larger = stronger (90 = exactly orthogonal planes);
                          # 45 is the minimum useful value. Paper used 45 / 60.
W_TORSAL         = 1.0
W_TANGLE         = 2.0    # paper Table 2: w_angle = 2
W_ORTH_TORSAL    = 0.01   # paper Table 2: w_LCOrth = 0.01 here -- two orders
                          # below stage 2 on purpose. Holding the angle
                          # constraint hard at this stage drags l off
                          # unit(cu x cv) instead of letting the radii move.
RECOMPUTE_STEPS  = 25     # paper Table 2 "RS": re-solve the analytic torsal
                          # directions every N steps, so the optimisation does
                          # not stall on degenerate directions.
FINAL_PLAIN      = 10     # closing iterations with w_angle = 0
# ─────────────────────────────────────────────────────────────────────────────

state.angle_torsal = ANGLE_TORSAL_DEG

setup_torsal_optimizer(state, w_lc=W_LC, w_orth=W_ORTH_TORSAL,
                        w_torsal=W_TORSAL, w_tangle=W_TANGLE)
run_torsal_optimizer(state, max_iter=TORSAL_MAX_ITER,
                      recompute_steps=RECOMPUTE_STEPS,
                      final_plain_steps=FINAL_PLAIN)

In [49]:
from moebius.utils.mesh_utils import vec_dot
from moebius.pipeline import torsal_field_per_quad

l_t, nt1, nt2 = state.torsal_opt.unpack("l", "nt1", "nt2")
l_t = l_t.reshape(state.u_sample_num, state.v_sample_num, 3)

# t_k = u_k·du + v_k·dv in the SURFACE tangent basis of each quad
# (du = v2-v0, dv = v1-v3), the same convention as energies.Torsal.
t1_n, t2_n = torsal_field_per_quad(state)

nt1 = nt1.reshape(-1, 3)
nt2 = nt2.reshape(-1, 3)
torsal_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(nt1, nt2)), 0, 1)))

# Angle between the two torsal directions — near 0 means a degenerate frame,
# which is what makes the remesher produce singularities.
t_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(t1_n, t2_n)), 0, 1)))
print(f"angle(t1,t2)   min {t_angle.min():.1f}°  mean {t_angle.mean():.1f}°  max {t_angle.max():.1f}°")
print(f"degenerate (<5°): {int((t_angle < 5).sum())} / {len(t_angle)} quads")

# Face barycenters, arrows scaled to ~1/4 of the mean face diagonal
bary = (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]] + V[F[:, 3]]) / 4
diag = np.mean(np.linalg.norm(V[F[:, 2]] - V[F[:, 0]], axis=1))
s = diag / 4

# Each face contributes one centred segment  [bary - s*t, bary + s*t]
def _torsal_curves(bary, t, scale):
    F_cnt = len(bary)
    verts = np.empty((2 * F_cnt, 3))
    verts[0::2] = bary - scale * t
    verts[1::2] = bary + scale * t
    edges = np.stack([np.arange(0, 2*F_cnt, 2),
                      np.arange(1, 2*F_cnt, 2)], axis=1)
    return verts, edges

v1_net, e1_net = _torsal_curves(bary, t1_n, s)
v2_net, e2_net = _torsal_curves(bary, t2_n, s)

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "torsal_angle", torsal_angle, defined_on="faces")
kv.add_scalar_quantity("surface", "t1_t2_angle",  t_angle,      defined_on="faces")
kv.add_vector_quantity("surface", "l", l_t.reshape(-1, 3), defined_on="vertices")

kv.register_curve_network("torsal_t1", v1_net, e1_net, color=(0.9, 0.2, 0.1), radius=0.002)
kv.register_curve_network("torsal_t2", v2_net, e2_net, color=(0.1, 0.3, 0.9), radius=0.002)
kv.show()

angle(t1,t2)   min 21.6°  mean 36.9°  max 60.6°
degenerate (<5°): 0 / 576 quads
[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 4 — Remeshing

`quadRemesher` takes a **triangle** mesh plus a **frame field** (two directions
per face) and integrates the field into a quad mesh whose edges follow it.
`export_frame_field` writes what it needs into `state.save_path`:

| file | contents |
|---|---|
| `remesh_input.obj` | the sampling grid, **triangulated** (quads are rejected) |
| `D1.dat`, `D2.dat` | one unit torsal direction per constrained triangle, `--format vec` |
| `indices.dat` | which triangles those rows belong to (`--indices`) |

A torsal direction lies in the plane of its quad's two **diagonals**. Where the
quad is non-planar neither of its two triangles lies in that plane, so the
remesher's in-plane projection rotates the direction. `export_frame_field`
therefore constrains only **one triangle per quad** — whichever needs the
smaller rotation — and lets the remesher interpolate the rest
(`one_frame_per_quad=True`, the default). On `CMC` at 15² sampling that took the
rotation from 5.2°/49° (mean/max) down to 3.7°/28° and the singularity count
from 11 to 9. Pass `one_frame_per_quad=False` to constrain both triangles.

**Quad density is `--gradient H`** (default 40): *higher = denser*, relative to
the bounding box. `--constraints F` keeps only a fraction `F` of the faces as
hard constraints; lowering it to ~0.1 lets the solver smooth the field and
usually gives fewer singularities.

The remesher runs two ways, both via `run_remesher(..., batch=...)`:

- `batch=True` — headless: integrate, extract, write `<name>_Remeshed.obj`, exit.
- `batch=False` — open its viewer: `+`/`-` change the gradient live, `R`
  previews the quads, `S` saves. Use this to find a good `H` by eye, then set
  `GRADIENT` below to that value and re-run in batch so the state holds it.

In [50]:
# ── export the frame field ────────────────────────────────────────────────────
from moebius.pipeline import (
    check_remesher, export_frame_field, run_remesher,
    load_remeshed_surface, remesh_quality, sweep_gradient,
)

# Found automatically at ../QuadRemesher/bin/quadRemesher or
# ../QuadRemesher/Quad_Remesher/build/quadRemesher; override if it lives elsewhere:
# state.remesher_bin = "/path/to/quadRemesher"
check_remesher(state.remesher_bin or None)

ONE_FRAME_PER_QUAD = True   # constrain 1 of each quad's 2 triangles (see above)

field = export_frame_field(state, one_frame_per_quad=ONE_FRAME_PER_QUAD)
field

Remesher OK: /home/anthony/KAUSTLocal/QuadRemesher/Quad_Remesher/build/quadRemesher
Exported 625 vertices, 1152 triangles -> /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0
  576 frames on 1 of the 2 triangles of each of 576 quads
  |angle(t1,t2)|  min 21.6 deg  mean 36.9 deg  max 60.6 deg
  in-plane rotation on the constrained triangles: mean 0.1 deg, max 0.8 deg


{'obj': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/remesh_input.obj',
 'd1': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/D1.dat',
 'd2': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/D2.dat',
 'indices': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/indices.dat',
 'n_vertices': 625,
 'n_triangles': 1152,
 'n_frames': 576,
 'n_quads': 576,
 'angle_t1_t2_deg': {'min': 21.554236936018256,
  'mean': 36.92525658894059,
  'max': 60.63779460912226},
 'rotation_deg': {'mean': 0.1161268369632243,
  'max': 0.7649119610615076,
  'n_over_15': 0}}

In [51]:
# ── check what was exported: read the files back and draw them ───────────────
# If the field looks wrong HERE, the remesher cannot fix it — go back to
# stage 3b rather than tuning the gradient.
V_tri, F_tri = read_obj(field["obj"])
F_tri = np.array(F_tri, dtype=np.int32)
d1 = np.loadtxt(field["d1"])
d2 = np.loadtxt(field["d2"])

# Rows refer to the triangles listed in indices.dat, or to all of them
if "indices" in field:
    tri_idx = np.loadtxt(field["indices"], dtype=int).ravel()
else:
    tri_idx = np.arange(len(F_tri))
print(f"{len(V_tri)} vertices, {len(F_tri)} triangles, "
      f"{len(d1)} frames on {len(tri_idx)} constrained triangles")
assert len(d1) == len(d2) == len(tri_idx)

tri_bary = V_tri[F_tri[tri_idx]].mean(axis=1)
s_tri = np.mean(np.linalg.norm(V_tri[F_tri[:, 1]] - V_tri[F_tri[:, 0]], axis=1)) / 3

def _seg_net(p, d, scale):
    n = len(p)
    verts = np.empty((2 * n, 3))
    verts[0::2] = p - scale * d
    verts[1::2] = p + scale * d
    edges = np.stack([np.arange(0, 2*n, 2), np.arange(1, 2*n, 2)], axis=1)
    return verts, edges

# Which triangles carry a constraint
constrained = np.zeros(len(F_tri))
constrained[tri_idx] = 1.0

kv.register_surface_mesh("remesh_input", V_tri, F_tri, show_edges=True)
kv.add_scalar_quantity("remesh_input", "constrained", constrained, defined_on="faces")
kv.register_curve_network("D1", *_seg_net(tri_bary, d1, s_tri), color=(0.9, 0.2, 0.1), radius=0.0015)
kv.register_curve_network("D2", *_seg_net(tri_bary, d2, s_tri), color=(0.1, 0.3, 0.9), radius=0.0015)
kv.show()

625 vertices, 1152 triangles, 576 frames on 576 constrained triangles
[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


In [52]:
# ── run the remesher (headless) ──────────────────────────────────────────────
GRADIENT    = 40.0    # higher = denser quads
CONSTRAINTS = 0.1     # fraction of faces kept as hard constraints (0 < f <= 1)
EXTRACTOR   = "qex"   # 'qex' (robust at seams) or 'grid'

res = run_remesher(state, gradient=GRADIENT, constraints=CONSTRAINTS,
                   extractor=EXTRACTOR, batch=True)

if res["returncode"] == 0:
    remesh_quality(state)
    kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh,
                             show_edges=True)
    kv.add_vector_quantity("remeshed", "l", state.l_remesh, defined_on="vertices")
    kv.show()

/home/anthony/KAUSTLocal/QuadRemesher/Quad_Remesher/build/quadRemesher /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/remesh_input.obj --field data --d1 /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/D1.dat --d2 /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/D2.dat --format vec --defined_on faces --indices /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/indices.dat --gradient 40 --constraints 0.1 --extractor qex --out /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0 --name Tunel --batch 


  Thinned to 58 constraint faces (10% of valid, farthest-point sampled)
Interpolating Frame Field from 58 constraints...
Deforming Mesh for Anisotropy...
Precomputing ARAP terms
Precomputing SMOOTH terms
Preconditioning done.
Converting to Cross Field...
Valid deformed constraints: 1152 / 1152
Running nrosy with 1152 constraints...
Singularities from nrosy (S field):
  Total 

In [54]:
# ── too coarse / too dense? sweep the gradient and pick one ──────────────────
# Each run overwrites the OBJ, so the state ends up holding the LAST value —
# re-run the cell above at your chosen GRADIENT afterwards.
rows = sweep_gradient(state, [20, 40, 80], constraints=CONSTRAINTS,
                      extractor=EXTRACTOR)

print()
for r in rows:
    if r.get("n_faces"):
        print(f"  gradient {r['gradient']:>5} -> {r['n_faces']:>5} faces  "
              f"({100*r['quad_frac']:.0f}% quads)  mean radius {r['mean_radius']:.4f}")


-- gradient 20 ------------------------------
Loaded /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/Tunel_Remeshed.obj
  139 vertices, 107 faces  107x4-gon
gradient 20  ->  107 faces (107 quads, 100.0%), 139 vertices, mean face radius 0.0502

-- gradient 40 ------------------------------
Loaded /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/Tunel_Remeshed.obj
  552 vertices, 489 faces  489x4-gon
gradient 40  ->  489 faces (489 quads, 100.0%), 552 vertices, mean face radius 0.0253

-- gradient 80 ------------------------------
Loaded /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/Tunel/experiment0/Tunel_Remeshed.obj
  2225 vertices, 2096 faces  2096x4-gon
gradient 80  ->  2096 faces (2096 quads, 100.0%), 2225 vertices, mean face radius 0.0127

  gradient    20 ->   107 faces  (100% quads)  mean radius 0.0502
  gradient    40 ->   489 faces  (100% quads)  mean radius 0.0253
  gradient    80 ->  2096 faces  (100% quads)  mean ra

In [ ]:
# ── or tune it by eye in the remesher's own viewer ───────────────────────────
# Blocks until you close the window. '+'/'-' = denser/coarser (x1.5 per press),
# 'R' = preview the quad mesh, 'S' = extract and save it, '6' = seams and
# singularities. Note the gradient you settle on and put it in GRADIENT above.
# res = run_remesher(state, gradient=GRADIENT, constraints=CONSTRAINTS,
#                    extractor=EXTRACTOR, batch=False)

In [ ]:
# ── load a previously remeshed OBJ ───────────────────────────────────────────
# load_remeshed_surface(state)                       # <save_path>/<name>_Remeshed.obj
# load_remeshed_surface(state, "/path/to/other_Remeshed.obj")
# remesh_quality(state)
# kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh, show_edges=True)
# kv.show()

---
## 5 — Post-optimisation

In [55]:
# ── parameters ────────────────────────────────────────────────────────────────
POSTOPT_MAX_ITER = 60
W_SPHERE   = 1.0
W_SUPPORT  = 1e-1
W_REG      = 1e-2
# ─────────────────────────────────────────────────────────────────────────────

setup_postopt_optimizer(state,
                         w_sphere=W_SPHERE,
                         w_support=W_SUPPORT,
                         w_reg=W_REG)
run_postopt_optimizer(state, max_iter=POSTOPT_MAX_ITER)

Mesh |V|=2225, |F|=2096, |E|=4320
Adding quadratic fairness for 2021 vertices with 4 neighbors
Adding Laplacian fairness for 204 
Thread pool: 5 workers (terms=5, cpus=32)
Starting fixed-damping LM optimization (max_iter=60, tol=1e-08)
Optimization failed at iteration 0: The numpy.ndarray conversion method __array__() was called on traced array with shape float32[3]
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerArrayConversionError

 WEIGHTS REPORT
Term                    Weight
------------------------------
SphereFit           1.0000e+00
SupportPlanarity    1.0000e-01
RegFaces            1.0000e-02
QuadFairness_v      9.0000e-04
Lap_Fairness_v      9.0000e-04
 Total Iterations: 0
 Time per Iteration: 0.0000 seconds
 ENERGY REPORT
Term               Energy      Normalized  Contrib.
---------------------------------------------------
SphereFit    1.849155e-01    2.205577e-05    100.0%
 FINAL ENERGY SUMMARY
  No optimization performed yet.



In [14]:
# ── visualise post-opt result ─────────────────────────────────────────────────
V_opt = state.postopt.unpack("v").reshape(-1, 3)
A, B, C = state.postopt.unpack("A", "B", "C")
B_r = B.reshape(-1, 3)
centers, radii = sphere_centers_radii_from_params(A, B_r, C)

# Spherical panels
pV, pF = spherical_panels_from_mesh(V_opt, state.F_remesh_quads, centers, radii, n_arc=12)

kv.register_surface_mesh("optimised_mesh",  V_opt,  state.F_remesh_quads)
kv.register_surface_mesh("sphere_panels",   pV,     pF)
kv.add_scalar_quantity("optimised_mesh", "sphere_radius", radii, defined_on="faces")
kv.show()

[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 6 — Save results

In [ ]:
save_results(state)
print("Saved to:", state.save_path)

---
## Utilities — reload & inspect

```python
# Reload a saved state
state = load_state("../notebooks/out/CMC/experiment0/state.pickle")

# Plot energy history
plt.figure(figsize=(8, 4))
plt.semilogy(state.lc_opt.energy,     label="LC")
plt.semilogy(state.torsal_opt.energy, label="Torsal")
plt.legend(); plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.grid()
```